Imported the PyPDFDirectoryLoader tool, which is used for loading and reading PDF document files

In [34]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

Used the PyPDFDirectoryLoader tool to load all the documents in the documents directory and then verified that it is working correctly by printing the number of loaded pages, the page content (to a defined limit) and the metadata

In [ ]:
# Loading all PDF files from the documents directory
loader = PyPDFDirectoryLoader("../documents")
documents = loader.load()

# Checking how many pages were loaded
print(f"Number of pages loaded: {len(documents)}")

# Previewing the beginning of the first page
print(documents[0].page_content[:100])

# Displaying metadata for the first page
print(documents[0].metadata)

Number of pages loaded: 1048
Service delivery
Consolidated  
HIV guidelines
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Macintosh)', 'creationdate': '2026-07-22T09:54:23+02:00', 'moddate': '2026-07-22T09:54:30+02:00', 'trapped': '/False', 'source': '..\\documents\\document1.pdf', 'total_pages': 50, 'page': 0, 'page_label': 'A'}


Imported and used the RecursiveCharacterTextSplitter for chunking the text in all the documents. Defined variables to set the total chunking size and the amount of chunk overlap allowed

In [ ]:
# Importing the text splitter used to divide documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter #RecursiveCharacterTextSplitter splits text by an ordered list of characters.

# Configuring how the document text will be divided
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Setting the maximum size of each chunk
    chunk_overlap=200,     # Chunk overlap is used to preserve context between neighboring chunks
    add_start_index=True   # Used to store each chunk's starting position
)

# Spliting the loaded documents into smaller chunks
splits = text_splitter.split_documents(documents)

# Checking how many chunks were created
print(f"Number of chunks: {len(splits)}")

Number of chunks: 3863


Imported the `HuggingFaceEmbeddings` class for vectorization. Initialized the embedding pipeline using the `all-MiniLM-L6-v2` model from SentenceTransformers to generate vector representations of the chunked documents.

In [ ]:
# Importing the LangChain-compatible Hugging Face embeddings wrapper
from langchain_huggingface import HuggingFaceEmbeddings

# Loading the MiniLM embedding model for the RAG pipeline
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Imported Chroma from LangChain to set up the vector store

In [ ]:
# Importing the Chroma vector store
from langchain_chroma import Chroma

# Creating a new Chroma vector database
vector_store = Chroma(
    collection_name="rag_documents",
    embedding_function=embeddings,
    persist_directory="../chroma_db"
)

print("Documents currently in Chroma:", vector_store._collection.count())

Documents currently in Chroma: 15452


Splitting the text chunks into batches and looped through them to upload to Chroma

In [ ]:
# Setting the chunk size for batch processing to manage memory efficiently
batch_size = 50
total_chunks = len(splits)

# Looping through all document splits in steps specified by batch_size
for start in range(0, total_chunks, batch_size):
    end = min(start + batch_size, total_chunks)

    # Extracting the current batch of text splits
    batch = splits[start:end]

    # Pushing the current batch of documents into the persistent Chroma database
    vector_store.add_documents(
        documents=batch
    )

    print(
        f"Indexed {end}/{total_chunks} chunks "
        f"({end / total_chunks * 100:.1f}%)"
    )

print(
    "Documents currently in Chroma:",
    vector_store._collection.count()
)

Indexed 50/3863 chunks (1.3%)
Indexed 100/3863 chunks (2.6%)
Indexed 150/3863 chunks (3.9%)
Indexed 200/3863 chunks (5.2%)
Indexed 250/3863 chunks (6.5%)
Indexed 300/3863 chunks (7.8%)
Indexed 350/3863 chunks (9.1%)
Indexed 400/3863 chunks (10.4%)
Indexed 450/3863 chunks (11.6%)
Indexed 500/3863 chunks (12.9%)
Indexed 550/3863 chunks (14.2%)
Indexed 600/3863 chunks (15.5%)
Indexed 650/3863 chunks (16.8%)
Indexed 700/3863 chunks (18.1%)
Indexed 750/3863 chunks (19.4%)
Indexed 800/3863 chunks (20.7%)
Indexed 850/3863 chunks (22.0%)
Indexed 900/3863 chunks (23.3%)
Indexed 950/3863 chunks (24.6%)
Indexed 1000/3863 chunks (25.9%)
Indexed 1050/3863 chunks (27.2%)
Indexed 1100/3863 chunks (28.5%)
Indexed 1150/3863 chunks (29.8%)
Indexed 1200/3863 chunks (31.1%)
Indexed 1250/3863 chunks (32.4%)
Indexed 1300/3863 chunks (33.7%)
Indexed 1350/3863 chunks (34.9%)
Indexed 1400/3863 chunks (36.2%)
Indexed 1450/3863 chunks (37.5%)
Indexed 1500/3863 chunks (38.8%)
Indexed 1550/3863 chunks (40.1%)
Inde

Tested the vector database by using a sample query about HIV testing. The script retrieves the top 3 most relevant document chunks through similarility matching.

In [ ]:
# Testing semantic retrieval using a sample question
query = "What are the recommendations for HIV testing?"

# Retrieving the 3 most relevant document chunks based on semantic similarity
results = vector_store.similarity_search(
    query,
    k=3 # Used so that only the top 3 relevant chunks are returned
)

# Displays the retrieved chunks and their information about their sources
for i, doc in enumerate(results, 1):
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Content:", doc.page_content[:500])

Source: ..\documents\document5.pdf
Page: 31
Content: admission. For people who do not know their HIV 
status, WHO recommends that, in settings with a 
high burden of HIV, provider-initiated HIV testing 
and counselling be offered to all people presenting 
for care in all health-care settings. In settings with 
a low burden of HIV, people with conditions that 
could indicate HIV infection should be offered testing 
(72). Consideration should be given to making HIV 
testing services for inpatients available on evenings 
and weekends in all areas of 
Source: ..\documents\document5.pdf
Page: 31
Content: admission. For people who do not know their HIV 
status, WHO recommends that, in settings with a 
high burden of HIV, provider-initiated HIV testing 
and counselling be offered to all people presenting 
for care in all health-care settings. In settings with 
a low burden of HIV, people with conditions that 
could indicate HIV infection should be offered testing 
(72). Consideration should b

Imported the required classes from the transformers library to set up the LLM pipeline. Loaded the 'Qwen/Qwen3-4B' model and the 'AutoTokenizer'.

In [ ]:
# Importing the tokenizer and language model classes
from transformers import AutoTokenizer, AutoModelForCausalLM

# Specifying the Qwen3-4B language model
model_name = "Qwen/Qwen3-4B"

# Loading the AutoTokenizer, which is used to convert text into tokens
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Loading the Qwen3-4B model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

# Placing the model weights on the CPU execution context
model = model.to("cpu")

print("Qwen3-4B loaded successfully on CPU!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3-4B loaded successfully on CPU!


Integrated the Qwen model into LangChain by wrapping it in a Hugging Face text-generation pipeline.


In [ ]:
# Importing Hugging Face's text-generation pipeline and LangChain's wrapper for Hugging Face pipelines
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Creating the text-generation pipeline using Qwen3-4B
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False, # Disabling sampling for stable output
    repetition_penalty=1.1, # Added penalty to prevent phrase repetition
    return_full_text=False # Applied to exclude the prompt from the output
)

# Wrapping the Hugging Face pipeline so it can be used with LangChain
llm = HuggingFacePipeline(
    pipeline=generation_pipeline
)

Converted the Chroma vector store into a LangChain retriever component, allowing it to run similarity searches and fetch the 3 most relevant document contexts.


In [ ]:
# Creating a retriever from the Chroma vector store for the 3 most relevant document chunks based on user questions.
retriever = vector_store.as_retriever(
    search_type="similarity", # Using vector distance similarity matching
    search_kwargs={"k": 3} # Targeting the top 3 closest matches for question answers
)

print("Retriever created successfully!")

Retriever created successfully!


Set up the context retrieval and prompt engineering workflow for our RAG pipeline.

In [ ]:
# Defining a sample question for testing purposes.
question = "What is recommended for HIV testing in settings with a high burden of HIV?"

# Retrieving the most relevant document chunks
results = retriever.invoke(question)

print(f"Retrieved {len(results)} document chunks.")

# Combining the retrieved document chunks into a single context string.
context = "\n\n".join(
    [
        f"Source: {doc.metadata.get('source')}\n"
        f"Page: {doc.metadata.get('page')}\n"
        f"Content:\n{doc.page_content}"
        for doc in results
    ]
)

# Setting the RAG prompt using the retrieved context and the user's question

prompt = f"""
You are a helpful assistant answering questions about HIV service delivery.

Answer the QUESTION using ONLY the information in the CONTEXT.

Give ONE concise answer consisting of 1-2 sentences.

STOP immediately after the answer and citation.

Do not repeat any sentence.
Do not repeat any citation.
Do not explain your reasoning.
Do not evaluate your own answer.
Do not mention these instructions.
Do not use outside knowledge.

If the context does not contain enough information to answer the question,
respond exactly:
I cannot answer this based on the provided documents.

For an answer supported by the context, use exactly this citation format:
(Source: document5.pdf, Page: 31)

Do not use any other citation format.
Do not include reference numbers such as (72).
Do not add a sources section.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

Retrieved 3 document chunks.


Implemented the final execution and post-processing steps for the RAG pipeline output

In [ ]:
# Generating the answer by passing the prompt to the LLM
response = llm.invoke(prompt)

# Removing Qwen's internal thinking section (if present) to allow for a relatively proper answer
if "</think>" in response:
    response = response.split("</think>", 1)[1].strip()

# Keeping only the answer through the first citation
if "(Source:" in response:
    citation_end = response.find(")", response.find("(Source:"))

    if citation_end != -1:
        response = response[:citation_end + 1]

print("FINAL ANSWER:")
print(response.strip())

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINAL ANSWER:
In settings with a high burden of HIV, provider-initiated HIV testing and counselling should be offered to all people presenting for care in all healthcare settings (Source: document5.pdf, Page: 31)
